In [1]:
from attrackt.scripts import create_zarr, create_csv, correct_gt_with_st
from attrackt.scripts.motile import motile_infer
from attrackt.scripts.cumulate_scores import cumulate_scores

In [2]:
data_dir = "/home/ddon0001/PhD/data/cell_tracking_challenge/SUBMISSION/Fluo-N2DL-HeLa"
out_dir = "/home/ddon0001/PhD/experiments/attrackt"

In [6]:
correct_gt_with_st(
    silver_truth_dir_name=data_dir + "/01_ST/SEG/",
    gold_truth_dir_name=data_dir + "/01_GT/TRA/",
    combined_truth_dir_name=out_dir + "/Fluo-N2DL-HeLa/01_GT/TRA/",
)


correct_gt_with_st(
    silver_truth_dir_name=data_dir + "/02_ST/SEG/",
    gold_truth_dir_name=data_dir + "/02_GT/TRA/",
    combined_truth_dir_name=out_dir + "/Fluo-N2DL-HeLa/02_GT/TRA/",
)

INFO:root:Corrected gold truth segmentations using silver truth segmentations.
INFO:root:Corrected gold truth segmentations using silver truth segmentations.


In [8]:
# copy the man_track.txt files to the new directory
import shutil
man_track_file_names = [
    data_dir + "/01_GT/TRA/man_track.txt",
    data_dir + "/02_GT/TRA/man_track.txt",
]
out_track_file_names = [
    out_dir + "/Fluo-N2DL-HeLa/01_GT/TRA/man_track.txt",
    out_dir + "/Fluo-N2DL-HeLa/02_GT/TRA/man_track.txt",
]
for man_track_file, out_track_file in zip(man_track_file_names, out_track_file_names):
    shutil.copy(man_track_file, out_track_file)

In [3]:
detections_csv_file_name = f'{out_dir}/Fluo-N2DL-HeLa/attrackt_detections.csv'
sequence_names = [
    "01",
    "02",
]
test_sequence_names = [
    "02",
]
mask_dir_names = [
    out_dir + "/Fluo-N2DL-HeLa/01_GT/TRA",
    out_dir + "/Fluo-N2DL-HeLa/02_GT/TRA",
]
man_track_file_names = [
    out_dir + "/Fluo-N2DL-HeLa/01_GT/TRA/man_track.txt",
    out_dir + "/Fluo-N2DL-HeLa/02_GT/TRA/man_track.txt",
]
img_dir_names = [
    data_dir + "/01/",
    data_dir + "/02/",
]

In [10]:
create_csv(
    mask_dir_names=mask_dir_names,
    sequence_names=sequence_names,
    man_track_file_names=man_track_file_names,
    output_csv_file_name=detections_csv_file_name,
)

INFO:attrackt.scripts.create_csv:Loaded manual track file: /home/ddon0001/PhD/experiments/attrackt/Fluo-N2DL-HeLa/01_GT/TRA/man_track.txt
INFO:attrackt.scripts.create_csv:Processing detections...
INFO:attrackt.scripts.create_csv:Processing manual track TXT...
INFO:attrackt.scripts.create_csv:Generating CSV output...
INFO:attrackt.scripts.create_csv:Loaded manual track file: /home/ddon0001/PhD/experiments/attrackt/Fluo-N2DL-HeLa/02_GT/TRA/man_track.txt
INFO:attrackt.scripts.create_csv:Processing detections...
INFO:attrackt.scripts.create_csv:Processing manual track TXT...
INFO:attrackt.scripts.create_csv:Generating CSV output...
INFO:attrackt.scripts.create_csv:Saving output CSVs to: /home/ddon0001/PhD/experiments/attrackt/Fluo-N2DL-HeLa/attrackt_detections.csv
INFO:attrackt.scripts.create_csv:CSV creation complete.


In [11]:
container_path = out_dir + "/HeLa.zarr"

create_zarr(
    container_path=container_path,
    img_dir_names=img_dir_names,
    mask_dir_names=mask_dir_names,
    sequence_names=sequence_names,
    mapping_csv_file_name=detections_csv_file_name,
)

INFO:attrackt.scripts.create_zarr:Sequence '01': using 8639 mapping rows.
INFO:attrackt.scripts.create_zarr:Sequence '01': relabeled masks written. Unique labels: 266 -> 8640
INFO:attrackt.scripts.create_zarr:Sequence '02': using 25420 mapping rows.
INFO:attrackt.scripts.create_zarr:Sequence '02': relabeled masks written. Unique labels: 675 -> 25421
INFO:attrackt.scripts.create_zarr:Created/updated container at /home/ddon0001/PhD/experiments/attrackt/HeLa.zarr.


In [4]:
result_dir_name = "results-position-only"
args = {
    "num_nearest_neighbours": 10,
    "direction_candidate_graph": "backward",
    "pin_nodes": True,
    "use_edge_distance": True,
    "edge_embedding_exists": False,
    "use_different_weights_hyper": True,
    "voxel_size": {"x": 1.0, "y": 1.0},
    "test_csv_file_name": detections_csv_file_name,
    "sequence_names": test_sequence_names,
    "result_dir_name": result_dir_name,
    "verbose": False,
}

motile_infer(args)

INFO:attrackt.scripts.motile.motile_infer:Processing sequence: 02
INFO:motile_toolbox.candidate_graph.utils:Extracting nodes from points list
INFO:motile_toolbox.candidate_graph.compute_graph:Candidate nodes: 25420
INFO:motile_toolbox.candidate_graph.utils:Extracting candidate edges
100%|██████████| 92/92 [00:00<00:00, 224.70it/s]
INFO:attrackt.scripts.motile.motile_infer:Flipping edges (backward direction)
INFO:attrackt.scripts.motile.utils:====================
Candidate Graph Stats
INFO:attrackt.scripts.motile.utils:num_nodes: 25420
INFO:attrackt.scripts.motile.utils:num_edges: 252950
INFO:attrackt.scripts.motile.utils:====================
Track Graph Stats
INFO:attrackt.scripts.motile.utils:num_nodes: 25420
INFO:attrackt.scripts.motile.utils:num_edges: 1463909
INFO:attrackt.scripts.motile.utils:====================
Candidate Graph Stats
INFO:attrackt.scripts.motile.utils:mean_regular_edge_distance: 58.90307159221014
INFO:attrackt.scripts.motile.utils:std_regular_edge_distance: 31.39